In [18]:
from matplotlib import pyplot as plt
import os
import tensorflow as tf
import keras
import numpy as np
import pickle

Filter out invalid images

In [19]:
images_path = "training_images"

num_skipped = 0
for root, dirs, files in os.walk(images_path):
    for file in files:
        _, extension = os.path.splitext(file)
        if extension.lower() == ".jpg":
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "rb") as fobj:
                    is_jfif = tf.compat.as_bytes("JFIF") in fobj.peek(10)
            except:
                is_jfif = False

            if not is_jfif:
                num_skipped += 1
                os.remove(filepath)

print("Deleted %d images" % num_skipped)


Deleted 0 images


Load the image datasets

In [20]:
training_dataset = keras.preprocessing.image_dataset_from_directory(
    images_path,
    labels='inferred',
    label_mode='categorical',
    batch_size=32,
    image_size=(150, 150),
    validation_split=0.1,
    subset="training",
    seed=1
)

validation_dataset = keras.preprocessing.image_dataset_from_directory(
    images_path,
    labels='inferred',
    label_mode='categorical',
    batch_size=32,
    image_size=(150, 150),
    validation_split=0.1,
    subset="validation",
    seed=1
)

class_names = training_dataset.class_names
print(class_names)

Found 14689 files belonging to 8 classes.
Using 13221 files for training.
Found 14689 files belonging to 8 classes.
Using 1468 files for validation.
['Banana', 'Capsicum', 'Cheese', 'Cucumber', 'Eggs', 'Milk', 'Tomato', 'Yogurt']


Build, compile and train model

In [21]:
# add more layers to improve accuracy
model = keras.Sequential([
    keras.Input(shape=(150, 150, 3)),
    # normalising the layers
    keras.layers.Rescaling(1./255),
    # https://www.tensorflow.org/tutorials/images/data_augmentation, https://www.tensorflow.org/guide/keras/preprocessing_layers
    # to improve accuracy
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomBrightness(0.1),
    keras.layers.RandomContrast(0.1),
    keras.layers.RandomRotation(0.2),

    # layers
    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(8, activation="softmax")
])



In [22]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    training_dataset,
    validation_data=validation_dataset,
    # more epochs (have the model see more images)
    # 20 epochs took 25mins accuracy 0.1444
    # 50 epochs took 2+ hours hit plateau at like epcoh 7? accuracy always 0.1368
    epochs=50
)

Epoch 1/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 127s 304ms/step - accuracy: 0.1319 - loss: 2.0944 - val_accuracy: 0.1226 - val_loss: 2.0785
Epoch 2/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 144s 347ms/step - accuracy: 0.1355 - loss: 2.0789 - val_accuracy: 0.1233 - val_loss: 2.0662
Epoch 3/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 137s 332ms/step - accuracy: 0.1377 - loss: 2.0783 - val_accuracy: 0.1410 - val_loss: 1.9967
Epoch 4/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 142s 343ms/step - accuracy: 0.1358 - loss: 2.0855 - val_accuracy: 0.1192 - val_loss: 2.0454
Epoch 5/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 146s 353ms/step - accuracy: 0.1367 - loss: 2.0769 - val_accuracy: 0.1226 - val_loss: 2.0787
Epoch 6/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 150s 363ms/step - accuracy: 0.1368 - loss: 2.0778 - val_accuracy: 0.1226 - val_loss: 2.0787
Epoch 7/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 147s 356ms/step - accuracy: 0.1368 - loss: 2.0778 - val_accuracy: 0.1226 - val_loss: 2.0788
Epoch 8/50
414/414 ━━━━━━━━━━━━━━━━━━━━ 147s 355ms/step - accuracy: 0.1368 -

Save model (so don't have to retrain everytime)

In [23]:
model.evaluate(validation_dataset)
model.save("grocery_image_model.keras")

46/46 ━━━━━━━━━━━━━━━━━━━━ 5s 119ms/step - accuracy: 0.1226 - loss: 2.0788


Load text and image model, do prediction

In [24]:
text_model = tf.keras.models.load_model("grocery_classifier.keras")

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    label_map = pickle.load(f)

In [25]:
def predict_product_from_image(img_path):
    img = keras.preprocessing.image.load_img(
        img_path, target_size=(150, 150)
    )
    img_array = keras.preprocessing.image.img_to_array(img)
    # chatgpt: preprocessing here as well to ensure it is consisten to avoid misclassification
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)
    predicted_class = class_names[np.argmax(preds)]

    return predicted_class

In [26]:
def predict_category(item_name: str) -> str:
    input_tensor = tf.constant([item_name])
    pred_probs = text_model.predict(input_tensor)
    label_index = int(tf.argmax(pred_probs, axis=1)[0])
    category = label_map[label_index]
    return category

In [27]:
def add_item_from_image(img_path):
    product = predict_product_from_image(img_path)
    category = predict_category(product)

    print("Item:", product)
    print("Category:", category)

    return product, category

In [28]:
if __name__ == "__main__":
    test_image_path = "training_images/Banana/banana.10.jpg"

    product, category = add_item_from_image(test_image_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step
Item: Capsicum
Category: snacks & sweets


In [29]:
# inaccurate put in an image of a banana and it got recognised as cheese and put into bread and grains category
# going to try adjust the model- change layers and epochs?